# A second judge: is the relevance measure a property of the replies or of the judge?

FairFewShot.ipynb found that the few-shot 3B, told to write short replies, gets a relevance position of 1.27: more predictable from the conversation than my real replies. Two readings: (a) the measure rewards short, on-topic, generic replies over real ones, whoever judges; (b) the judge, Qwen2.5-7B, shares a family with the 3B and prefers its way of writing.

Here **exactly the same replies** (every set scored so far: floor, my real replies, base, adapters, retrieval, few-shot with and without instructions, adapter + retrieval) are scored again by a judge from another family, **Llama-3.1-8B-Instruct** (4 bits), with the same contexts (6 turns), the same measure and the same bootstrap. Losses in nats aren't comparable across judges (different tokenizers), so the comparison is on the relevance position (0 = my reply to another conversation, 1 = my real reply), which is relative to each judge's own floor and ceiling.

Runs on the NAS GPU.

## Prediction (written before running)

Fede and Claude agree: the short 3B stays above 1 with Llama too, maybe somewhat less. It is more the measure than the judge's family: a short, on-topic reply is predictable for any model, and my real replies talk about things neither judge can know.

How it's read: if the short 3B's position with Llama stays above 1 (its interval above 1), the measure rewards generic on-topic replies whatever the judge; if it falls to the level of the other sets, it was the Qwen judge's preference for its own family.

In [1]:
import json, os, subprocess
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
PY = os.environ.get("TLM_PY", "uv run python")
JUDGE = "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit"
SETS = ["retrieval", "retrieval_valid", "adapter1.5b_t1", "adapter_rag3", "fewshot20_0.5b", "fewshot20_1.5b", "fewshot20_3b",
        "fewshot20_0.5b_short", "fewshot20_0.5b_style", "fewshot20_1.5b_short", "fewshot20_1.5b_style",
        "fewshot20_3b_short", "fewshot20_3b_style"]

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line and "█" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

## Score every set with Llama-3.1-8B

In [2]:
if not (R / "relevance_llama.json").exists():
    run(f"{PY} scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra {' '.join(SETS)} "
        f"--scorer {JUDGE} --cache experiments/results/relevance_nll_llama.json --out experiments/results/relevance_llama.json")

$ /opt/venv/bin/python scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra retrieval retrieval_valid adapter1.5b_t1 adapter_rag3 fewshot20_0.5b fewshot20_1.5b fewshot20_3b fewshot20_0.5b_short fewshot20_0.5b_style fewshot20_1.5b_short fewshot20_1.5b_style fewshot20_3b_short fewshot20_3b_style --scorer mlx-community/Meta-Llama-3.1-8B-Instruct-4bit --cache experiments/results/relevance_nll_llama.json --out experiments/results/relevance_llama.json
floor (other conv.): 100/1133 (0.3 min)
floor (other conv.): 200/1133 (0.5 min)
floor (other conv.): 300/1133 (0.8 min)
floor (other conv.): 400/1133 (1.0 min)
floor (other conv.): 500/1133 (1.2 min)
floor (other conv.): 600/1133 (1.5 min)
floor (other conv.): 700/1133 (1.7 min)
floor (other conv.): 800/1133 (1.9 min)
floor (other conv.): 900/1133 (2.1 min)
floor (other conv.): 1000/1133 (2.4 min)
floor (other conv.): 1100/1133 (2.6 min)
floor (other conv.): 1133/1133 (2.7 min)
base t1.0: 100/1133 (3.0 min)
base t1.0:

## The two judges side by side

In [3]:
qwen = json.load(open(R / "relevance_fewshot_fair.json"))["positions"] | json.load(open(R / "relevance_7b.json"))["positions"] \
       | json.load(open(R / "relevance_rag.json"))["positions"]
llama = json.load(open(R / "relevance_llama.json"))["positions"]
print(f"{'set':<24} {'Qwen2.5-7B':>22} {'Llama-3.1-8B':>22}")
for k in ["base t1.0", "adapter t1.0"] + SETS:
    q, l = qwen[k], llama[k]
    print(f"{k:<24} {q['value']:>6.2f} [{q['ci'][0]:5.2f},{q['ci'][1]:5.2f}] {l['value']:>6.2f} [{l['ci'][0]:5.2f},{l['ci'][1]:5.2f}]")
g = json.load(open(R / "relevance_llama.json"))["context_gain"]
print(f"\nLlama: floor {g['floor (other conv.)']['gain']:.3f}, my real reply {g['real (me)']['gain']:.3f} nats")
print("position: 0 = my reply to another conversation, 1 = my real reply")

set                                  Qwen2.5-7B           Llama-3.1-8B
base t1.0                 -0.57 [-0.80,-0.38]  -1.42 [-1.75,-1.13]
adapter t1.0               0.27 [ 0.09, 0.42]  -0.00 [-0.17, 0.15]
retrieval                  0.40 [ 0.22, 0.58]   0.06 [-0.12, 0.32]
retrieval_valid            0.37 [ 0.14, 0.64]   0.06 [-0.07, 0.24]
adapter1.5b_t1             0.33 [ 0.19, 0.54]   0.02 [-0.15, 0.29]
adapter_rag3               0.34 [ 0.17, 0.46]  -0.05 [-0.20, 0.13]
fewshot20_0.5b            -0.55 [-0.76,-0.38]  -1.46 [-1.79,-1.18]
fewshot20_1.5b            -0.11 [-0.29, 0.03]  -0.89 [-1.09,-0.69]
fewshot20_3b               0.27 [ 0.07, 0.46]  -0.50 [-0.79,-0.22]
fewshot20_0.5b_short      -0.39 [-0.58,-0.20]  -1.03 [-1.29,-0.77]
fewshot20_0.5b_style      -0.34 [-0.54,-0.16]  -1.02 [-1.32,-0.75]
fewshot20_1.5b_short       0.30 [ 0.14, 0.49]  -0.21 [-0.40, 0.02]
fewshot20_1.5b_style       0.29 [ 0.14, 0.47]  -0.17 [-0.37, 0.10]
fewshot20_3b_short         1.27 [ 1.06, 1.52]   1.01 [ 0.8

## Conclusion

**Prediction (both):** the short 3B stays above 1 with Llama, maybe somewhat less. **Half right: it drops to my level (1.01 [0.85, 1.29]), not above it. And the second judge changes something more important: the adapters.**

| Set | Qwen2.5-7B | Llama-3.1-8B |
|---|---|---|
| Fine-tuned 0.5B | 0.27 [0.09, 0.42] | **-0.00 [-0.17, 0.15]** |
| Fine-tuned 1.5B | 0.33 [0.19, 0.54] | 0.02 [-0.15, 0.29] |
| Fine-tuned 0.5B + retrieval | 0.34 [0.17, 0.46] | -0.05 [-0.20, 0.13] |
| Retrieval (validation pool) | 0.37 [0.14, 0.64] | 0.06 [-0.07, 0.24] |
| Few-shot 3B, original | 0.27 [0.07, 0.46] | -0.50 [-0.79, -0.22] |
| Few-shot 3B, short | 1.27 [1.06, 1.52] | **1.01 [0.85, 1.29]** |
| Few-shot 3B, short + style | 1.29 [1.07, 1.52] | 1.04 [0.86, 1.31] |
| Base 0.5B | -0.57 [-0.80, -0.38] | -1.42 [-1.75, -1.13] |

(Llama's band: floor 0.799, my real reply 1.206 nats.)

- **The short 3B is a property of the replies, not of the Qwen family.** With a judge of another family it falls from 1.27 to 1.01, so part of the excess was family preference, but it still scores as high as my real replies. Short, on-topic replies from an instruction-tuned model look as "relevant" as the real ones to any model judge.
- **The adapters' relevance depends on the judge.** With Qwen they cover about a quarter to a third of the band; with Llama they sit at the floor (0.5B: -0.00, CI -0.17 to 0.15), as does retrieval (0.06). The ordering is similar (short 3B on top, base and small few-shot at the bottom), but "the adapter answers a little better than a reply of mine to another conversation" doesn't survive a change of judge.
- **Both judges agree on the big picture:** my real replies are clearly above a reply of mine to another conversation, the base model and the small few-shot models are well below it, and nothing trained on me gets far into the band.

**What it changes.** The relevance measure is judge-dependent in the middle of its range and can be matched by generic on-topic replies. The robust content claim is the weak one: the adapter doesn't demonstrably answer better than a reply of mine copied from another conversation. The claim "it uses the conversation a little" (decision 22) holds with one judge and not with the other.